In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import os
import omicverse as ov

import helper.plot_helper as hp


In [ ]:

import scanpy as sc
import pandas as pd
# Plotting
import matplotlib
import matplotlib.pyplot as plt
from optax.contrib import dadapt_adamw

import scanpy as sc
import numpy as np
import cellrank as cr
from cellrank.kernels import ConnectivityKernel

In [ ]:
hp.setMyStyle()
os.getcwd()

In [ ]:
os.chdir('D:\\lxk\\project\\jiangshucai20260506')

In [ ]:
cellname = 'myeloid'

In [ ]:
import h5py

with h5py.File("./h5ad/sft.h5ad", "r+") as f:
    if "uns/log1p" in f:
        del f["uns/log1p"]
adata = sc.read_h5ad('./h5ad/myeloid.h5ad')
adata = adata[~adata.obs["celltype"].isin(["Cycling myeloid"])].copy()
#adata = adata[~adata.obs["sample"].isin(["LT-c",'LZ-c'])].copy()

In [ ]:
adata.obs['sample'].value_counts()

In [ ]:
import os
import numpy as np
import scanpy as sc
import cellrank as cr
from cellrank.kernels import PseudotimeKernel

# 1. 下采样，去掉 Cycling myeloid
sampling_plan = {
    "TREM2+ TAM": 4000,
    "HLA-high APC": 3000,
    "Inflammatory TAM": 3000,
    "LYVE1+ resident TAM": 2000,
    "FCN1+ monocyte": 1800,
    "LRMDA+ migratory TAM": 747,
    "SPP1+ lipid TAM": 683,
    "MMP9+ osteoclast-like TAM": 500,
}

selected = []

for ct, n in sampling_plan.items():
    cells = adata.obs_names[adata.obs["celltype"] == ct].to_numpy()
    if len(cells) == 0:
        print(f"Warning: {ct} not found")
        continue
    if len(cells) > n:
        cells = np.random.choice(cells, n, replace=False)
    selected.extend(cells)

adata_cr = adata[selected].copy()

# 2. 重新算 neighbors
if "X_scVI" in adata_cr.obsm:
    sc.pp.neighbors(
        adata_cr,
        use_rep="X_scVI",
        n_neighbors=30,
    )
elif "X_pca" in adata_cr.obsm:
    sc.pp.neighbors(
        adata_cr,
        use_rep="X_pca",
        n_neighbors=30,
    )
else:
    sc.pp.pca(adata_cr)
    sc.pp.neighbors(
        adata_cr,
        use_rep="X_pca",
        n_neighbors=30,
    )

# 3. 重新算 diffmap
sc.tl.diffmap(adata_cr)

# 4. 设置 root cell：FCN1+ monocyte
mono_genes = [
    "FCN1", "S100A8", "S100A9", "VCAN",
    "LYZ", "LST1", "CTSS", "AIF1"
]
mono_genes = [g for g in mono_genes if g in adata_cr.var_names]

sc.tl.score_genes(
    adata_cr,
    gene_list=mono_genes,
    score_name="FCN1_monocyte_score",
)

root_group = "FCN1+ monocyte"
root_cells = adata_cr.obs_names[adata_cr.obs["celltype"] == root_group]

score = adata_cr.obs.loc[root_cells, "FCN1_monocyte_score"]
top_cells = score[score >= score.quantile(0.95)].index

# 在 monocyte signature 高的前 5% 中，选择靠近 FCN1+ monocyte 群中心的细胞
X_top = adata_cr[top_cells].obsm["X_umap"]
X_root = adata_cr[root_cells].obsm["X_umap"]

center = X_root.mean(axis=0)
dist = np.linalg.norm(X_top - center, axis=1)

root_cell_id = top_cells[np.argmin(dist)]
root_index = np.flatnonzero(adata_cr.obs_names == root_cell_id)[0]

adata_cr.uns["iroot"] = root_index

print(f"root cell: {root_cell_id}, index: {root_index}")

# 5. 重新跑 DPT
sc.tl.dpt(adata_cr)

# 6. CellRank PseudotimeKernel
pk = PseudotimeKernel(
    adata_cr,
    time_key="dpt_pseudotime"
)

pk.compute_transition_matrix(
    threshold_scheme="soft",
    frac_to_keep=0.3,
)

# 7. GPCCA
g = cr.estimators.GPCCA(pk)
g.compute_schur(n_components=8)
g.plot_spectrum(real_only=True)

In [ ]:
sc.settings.figdir = f"./fig/time/{cellname}/"  # 可改自己的目录
os.makedirs(sc.settings.figdir, exist_ok=True)

##umap图
sc.pl.umap(adata_cr, color=['celltype', 'dpt_pseudotime'],
           color_map='viridis',
           title=['celltype', 'DPT Pseudotime'],save = 'dpt_pseudotime.pdf')

## 通过流线可视化转移矩阵，注意，这不是RNA速率，是指转移矩阵在二维空间的嵌入
pk.plot_projection(
            color="celltype",
            recompute=True,
            basis="X_umap",
            title="",
            legend_loc="on data",
            alpha=0.25,
            linewidth=1,
            dpi=600,
            legend_fontsize = 7,
            #ax=ax,
            save = f"./fig/time/{cellname}/plot_projection.pdf"
        )


pk.plot_random_walks(
    seed=0,
    n_sims=100,
    start_ixs={"celltype": "Mono_Classical"},
    basis="X_umap",
    legend_loc="right",
    dpi=300,
    save = "random_walks.pdf"
)

In [ ]:
g.compute_macrostates(n_states=3, cluster_key="celltype")
g.plot_macrostates(which="all", basis='X_umap', legend_loc="right", s=100)

In [ ]:
g.predict_terminal_states(allow_overlap=True)
g.plot_macrostates(which="terminal", basis='X_umap')

In [ ]:

sc.settings.figdir = f"./fig/time/{cellname}"  # 可改自己的目录
os.makedirs(sc.settings.figdir, exist_ok=True)
g.compute_fate_probabilities()#计算每一个细胞，最终走到上面找到的某一个终点的概率是多少。
g.plot_fate_probabilities(legend_loc="right", basis="X_umap")
cr.pl.circular_projection(adata_cr, keys="celltype", legend_loc="right",dpi=500,save = "circular_projection.pdf")


In [ ]:
all_drivers = g.compute_lineage_drivers() #找出“所有分化路径的驱动”基因

#把乱糟糟的表达量数据拟合成平滑曲线。
model = cr.models.GAMR(adata_cr,
                       n_knots=6,#结点数。你可以理解为曲线的“关节”。6 个关节刚好能画出“上升-平稳-下降”这样的复杂动作。
                       smoothing_penalty=10.0)#平滑惩罚。数值越大，画出来的线越直、越硬；数值越小，线越抖。10.0 是标准硬度。

## 绘制每个路径的前60个驱动基因

In [ ]:
import os
import re
import scanpy as sc
import cellrank as cr

sc.settings.figdir = f"./fig/time/{cellname}/Genes/Top50"
os.makedirs(sc.settings.figdir, exist_ok=True)

def is_bad_gene(gene):
    bad_prefix = (
        "RPL", "RPS", "MRPL", "MRPS",
        "MT-",
        "HBA", "HBB", "HBD", "HBE", "HBG", "HBM", "HBQ", "HBZ",
        "MALAT1",
    )

    if gene.startswith(bad_prefix):
        return True

    # 常见无信息/难解释基因
    if gene in {
        "MALAT1", "NEAT1", "XIST", "TMSB4X", "TMSB10",
        "B2M", "ACTB", "GAPDH", "EEF1A1", "EEF1B2", "EEF1D",
        "FTL", "FTH1",
    }:
        return True

    # 预测基因、反义转录本、lncRNA 类
    if re.match(r"^(AC|AL|AP|LINC|MIR|SNHG|RNU|RNA|RP11|RP[0-9])", gene):
        return True

    if re.search(r"-AS[0-9]*$", gene):
        return True

    return False


cellNames = g.coarse_T.columns.tolist()

for cellName in cellNames:

    NE_drivers = g.compute_lineage_drivers(
        lineages=cellName,
        model=model
    )

    # 先过滤，再取 top50
    NE_drivers_filtered = NE_drivers[
        [not is_bad_gene(gene) for gene in NE_drivers.index]
    ]

    genes = NE_drivers_filtered.head(50).index.tolist()

    # 保存过滤后的 driver 表
    outdir = f"./fig/time/{cellname}/Genes/Top50/{cellName}"
    os.makedirs(outdir, exist_ok=True)
    NE_drivers_filtered.to_csv(f"{outdir}/{cellName}_filtered_drivers.csv")

    cr.pl.heatmap(
        adata_cr,
        model=model,
        lineages=cellName,
        return_models=False,
        genes=genes,
        cbar=False,
        time_key="dpt_pseudotime",
        show_fate_probabilities=False,
        figsize=(8, 16),
        save=f"{cellName}_top50_heatmap.pdf",
        dpi=500,
        show_all_genes=True,
        cluster_key="celltype",
        cluster_genes=False,
    )

    for gene in genes:
        cr.pl.gene_trends(
            adata_cr,
            model=model,
            genes=gene,
            same_plot=True,
            time_key="dpt_pseudotime",
            hide_cells=True,
            weight_threshold=(1e-3, 1e-3),
            dpi=500,
            save=f"./{cellName}/{gene}.pdf",
        )

In [ ]:
sc.settings.figdir = f"./fig/time/{cellname}/Genes/Top50"  # 可改自己的目录
os.makedirs(sc.settings.figdir, exist_ok=True)
cellNames = g.coarse_T.columns.tolist()# 每个宏观状态的名称
for cellName in cellNames:

    NE_drivers = g.compute_lineage_drivers(lineages=cellName,model=model)
    genes = NE_drivers.head(50).index.tolist()
    cr.pl.heatmap(
    adata_cr,
    model=model,
    lineages=cellName,
    return_models=False,
    genes=genes,
    cbar=False,
    time_key="dpt_pseudotime",
    show_fate_probabilities=False,
    figsize=(8, 16),
    save = f'{cellName}_top50_heatmap.pdf',
    dpi=500,
    show_all_genes=True, cluster_key="celltype", cluster_genes=False )

    for gene in genes:
        cr.pl.gene_trends(
        adata_cr,
        model=model,
        genes=gene,
        same_plot=True,
        #ncols=3,
        time_key="dpt_pseudotime",
        hide_cells=True,#隐藏那些点
        weight_threshold=(1e-3, 1e-3),
        dpi=500,
        save=f"./{cellName}/{gene}.pdf",
    )

In [ ]:
from cellrank.kernels import PseudotimeKernel
sc.settings.figdir = f"./fig/time/{cellname}/"  # 可改自己的目录
os.makedirs(sc.settings.figdir, exist_ok=True)

sc.tl.diffmap(adata)

sc.tl.dpt(adata)


pk = PseudotimeKernel(adata, time_key="dpt_pseudotime")
pk.compute_transition_matrix() #细胞概率表

In [ ]:
sc.settings.figdir = f"./fig/time/{cellname}/"  # 可改自己的目录
os.makedirs(sc.settings.figdir, exist_ok=True)

##umap图
sc.pl.umap(adata, color=['celltype', 'dpt_pseudotime'],
           color_map='viridis',
           title=['celltype', 'DPT Pseudotime'],save = 'dpt_pseudotime.pdf')

## 通过流线可视化转移矩阵，注意，这不是RNA速率，是指转移矩阵在二维空间的嵌入
pk.plot_projection(
            color="celltype",
            recompute=True,
            basis="X_umap",
            title="",
            legend_loc="on data",
            alpha=0.25,
            linewidth=1,
            dpi=600,
            legend_fontsize = 7,
            #ax=ax,
            save = f"./fig/time/{cellname}/plot_projection.pdf"
        )


pk.plot_random_walks(
    seed=0,
    n_sims=100,
    start_ixs={"celltype": "Mono_Classical"},
    basis="X_umap",
    legend_loc="right",
    dpi=300,
    save = "random_walks.pdf"
)

## 计算宏观状态
宏观状态：关键站点、终点

In [ ]:
g = cr.estimators.GPCCA(pk) #宏观状态
g.compute_schur(n_components=10)
g.plot_spectrum(real_only=True)

In [ ]:
g.compute_macrostates(n_states=5, cluster_key="celltype")
g.plot_macrostates(which="all", basis='X_umap', legend_loc="right", s=100)